# Lecture 1 Notebook

**Note:** In this example there is a lot of code. You are not expected to know how to read any of this yet. This is just a preview of the things you will see in the next few weeks. 


## This is a Jupyter Notebook

A Jupyter Notebook is a document that combines:

1. **Narrative:** The text describing your analysis
2. **Code:** The program that does the analysis
3. **Results:** The output of the program

The Jupyter environment was created by faculty here at Berkeley (Fernando Perez). These ideas are now in a lot of different technologies (e.g., Google Colab). 

## Our first example: analyzing the text of Little Women

*["Little Women"](https://en.wikipedia.org/wiki/Little_Women)* by Louisa May Alcott is the story of four sisters and their neighbor Laurie (Theodore Laurence).

To analyze it, we define a **function** that reads data from the Internet.

In [1]:
# A tiny program to download text from the web.
def read_url(url): 
    from urllib.request import urlopen 
    import re
    return re.sub('\\s+', ' ', urlopen(url).read().decode())

Here we download the book, which is hosted on the Data 8 textbook website.

In [2]:
little_women_url = 'https://www.inferentialthinking.com/data/little_women.txt'
little_women_text = read_url(little_women_url)
little_women_chapters = little_women_text.split('CHAPTER ')[1:]

## Tables

- A lot of data science is about transforming data. This is often in service of producing **tables**, a widely used data structure that we can more easily analyze our data with. 
- In this class you will use the `datascience` library (specifically created for this course) to manipulate and data.

In [ ]:
# datascience
from datascience import *


In [ ]:
# polars
import polars as pl
from IPython.display import display
import hvplot.polars


In [ ]:
# datascience
Table().with_column('Chapters', little_women_chapters)


In [ ]:
# polars
pl.DataFrame([
    pl.Series('Chapters', little_women_chapters)
])


## We will learn to visualize data

Most notebooks contain some setup for data manipulation and visualization. The details aren't important, but an LM can tell you about them.

In [ ]:
# datascience
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
# polars
import polars as pl
from IPython.display import display
import hvplot.polars


We can calculate the number of times Jo was mentioned in each chapter.

In [ ]:
# datascience
np.char.count(little_women_chapters, 'Jo')


In [ ]:
# polars
pl.Series('text', little_women_chapters).str.count_matches('Jo', literal=True)


How about the other characters? We could build a table.

In [ ]:
# datascience
# Counts of names in the chapters of Little Women

names = ['Amy', 'Beth', 'Jo', 'Meg', 'Laurie']
mentions = {name: np.char.count(little_women_chapters, name) for name in names}

counts = Table().with_columns([
        'Amy',    mentions['Amy'],
        'Beth',   mentions['Beth'],
        'Jo',     mentions['Jo'],
        'Meg',    mentions['Meg'],
        'Laurie', mentions['Laurie']
    ])
counts


In [ ]:
# polars
# Counts of names in the chapters of Little Women

names_pl = ['Amy', 'Beth', 'Jo', 'Meg', 'Laurie']
mentions_pl = {name: pl.Series('text', little_women_chapters).str.count_matches(name, literal=True) for name in names_pl}

counts_pl = pl.DataFrame([
    (mentions_pl['Amy']).alias('Amy'),
    (mentions_pl['Beth']).alias('Beth'),
    (mentions_pl['Jo']).alias('Jo'),
    (mentions_pl['Meg']).alias('Meg'),
    (mentions_pl['Laurie']).alias('Laurie')
])
counts_pl


We can accumulate the counts for each character to show the total in the book so far.

In [ ]:
# datascience
cumulative_counts = Table()
for label in counts.labels:
    cumulative_counts = cumulative_counts.with_column(label, np.cumsum(counts.column(label)))
cumulative_counts


In [ ]:
# polars
cumulative_counts_pl = pl.DataFrame([
    counts_pl.get_column(label).cum_sum().alias(label)
    for label in counts_pl.columns
])
cumulative_counts_pl


And then to study these numbers we can create a line chart of cumulative counts.

In [ ]:
# datascience
cumulative_counts.plot()
plt.title('Cumulative # of Times A Name Appears in Little Women')
plt.show()


In [ ]:
# polars
# matplotlib
for _col in cumulative_counts_pl.columns:
    plt.plot(cumulative_counts_pl.get_column(_col), label=_col)
plt.legend()
plt.title('Cumulative # of Times A Name Appears in Little Women')
plt.show()


Read the [book chapter](https://inferentialthinking.com/chapters/01/3/1/literary-characters/)!